# Phishing Website Detection (repaired and accuracy-tuned)

This version preserves the Random Forest classifier and wxPython GUI. It trains and validates using a stratified split of `training.xlsx`, then tunes Random Forest hyperparameters with 5-fold cross-validation on the training portion only. `feature_test.csv` is retained in the project for separate dataset checks, but is not mixed into model selection because its feature distribution differs from `training.xlsx`. The GUI accepts a URL and displays the model prediction. Keep the notebook and datasets in the same folder.

In [ ]:
import ipaddress
import re
import webbrowser
from urllib.parse import urlsplit

import tldextract

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wx

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)


In [ ]:
# The model was trained with these 10 input features, in this exact order.
FEATURE_COLUMNS = [
    "Having_IP",
    "URL_Length",
    "Shortening_Service",
    "Having_At_Symbol",
    "Double_Slash_Redirecting",
    "Prefix_Suffix",
    "Having_Sub_Domain",
    "SSLfinal_State",
    "Domain_Age",
    "Domain_End",
]


def importdata_train():
    """Load the labelled feature dataset used to train the model."""
    data = pd.read_excel("training.xlsx")
    data.columns = data.columns.astype(str).str.strip()

    required = ["Label"] + FEATURE_COLUMNS
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise ValueError(f"training.xlsx is missing columns: {missing}")

    data = data[required].copy()
    data = data.apply(pd.to_numeric, errors="raise")
    print("Training dataset shape:", data.shape)
    print(data.head())
    return data


In [ ]:
def importdata_test():
    """Load the separate labelled feature dataset used for evaluation."""
    data = pd.read_csv("feature_test.csv")
    data.columns = data.columns.astype(str).str.strip()

    required = ["Label"] + FEATURE_COLUMNS
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise ValueError(f"feature_test.csv is missing columns: {missing}")

    data = data[required].copy()
    data = data.apply(pd.to_numeric, errors="raise")
    print("Testing dataset shape:", data.shape)
    print(data.head())
    return data


In [ ]:
def splitdataset(data):
    """Separate the target label from the 10 model input features."""
    X = data[FEATURE_COLUMNS].to_numpy(dtype=int)
    y = data["Label"].to_numpy(dtype=int)
    return X, y


## URL validation and feature extraction for the GUI

The GUI first checks that an input has a recognized public domain suffix using the Public Suffix List (`tldextract`). An incomplete hostname such as `https://google` is rejected as an invalid/incomplete public website address before Random Forest prediction. This is a structure check, not proof that a domain is registered or safe. The supplied training data contains precomputed features; a URL alone cannot reveal domain age or reliably verify an SSL certificate, so those two features remain 0.


In [ ]:
SHORTENING_SERVICES = {
    "bit.ly", "tinyurl.com", "t.co", "goo.gl", "ow.ly",
    "is.gd", "buff.ly", "adf.ly", "bit.do", "cutt.ly",
}

# Use the bundled Public Suffix List snapshot; no network request is needed.
PUBLIC_SUFFIX_EXTRACTOR = tldextract.TLDExtract(suffix_list_urls=())


def validate_public_website_url(url):
    """Reject malformed URLs and hostnames without a recognized public suffix.

    This checks domain structure only. It does not prove that a domain is
    registered, reachable, safe, or non-phishing.
    """
    url = str(url).strip()
    if not url:
        raise ValueError("Please enter a URL.")

    normalized_url = url if re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", url) else "https://" + url
    try:
        parsed = urlsplit(normalized_url)
        hostname = (parsed.hostname or "").strip(".").lower()
        # Accessing .port also detects malformed port values.
        _ = parsed.port
    except ValueError:
        raise ValueError("Invalid URL format. Please enter a website such as example.com.")

    if not hostname:
        raise ValueError("Invalid URL: no hostname was provided.")

    # IP addresses are allowed through to the model (the model has Having_IP).
    try:
        ipaddress.ip_address(hostname)
        return normalized_url, parsed, hostname
    except ValueError:
        pass

    # A public website needs a recognized public suffix, e.g. .com or .co.uk.
    domain_info = PUBLIC_SUFFIX_EXTRACTOR(hostname)
    if not domain_info.suffix or not domain_info.domain:
        raise ValueError(
            "INVALID / INCOMPLETE WEBSITE ADDRESS: '" + hostname + "' does not have a recognized public domain suffix. "
            "Enter a complete public website address, such as google.com. This is not a phishing verdict."
        )

    # Reject whitespace and malformed DNS labels.
    labels = hostname.split(".")
    if any(not label or len(label) > 63 or label.startswith("-") or label.endswith("-") for label in labels):
        raise ValueError("Invalid hostname format. Check the domain name and try again.")

    return normalized_url, parsed, hostname


def extract_url_features(url):
    """Return the 10 features in the same order used by the trained model."""
    normalized_url, parsed, hostname = validate_public_website_url(url)

    try:
        ipaddress.ip_address(hostname)
        having_ip = 1
    except ValueError:
        having_ip = 0

    url_length = int(len(str(url).strip()) > 54)
    shortening_service = int(hostname in SHORTENING_SERVICES)
    having_at_symbol = int("@" in str(url))

    # A second // after the scheme can indicate a redirect-style URL.
    scheme_end = normalized_url.find("//") + 2
    double_slash_redirecting = int("//" in normalized_url[scheme_end:])

    prefix_suffix = int("-" in hostname)
    host_parts = [part for part in hostname.split(".") if part]
    if host_parts and host_parts[0] == "www":
        host_parts = host_parts[1:]
    having_sub_domain = int(len(host_parts) > 2)

    # This checks for an https scheme only; it does NOT validate the certificate.
    sslfinal_state = int(parsed.scheme.lower() == "https")

    # These require WHOIS/domain-history data and cannot be determined from a URL string alone.
    domain_age = 0
    domain_end = 0

    return np.array([[
        having_ip,
        url_length,
        shortening_service,
        having_at_symbol,
        double_slash_redirecting,
        prefix_suffix,
        having_sub_domain,
        sslfinal_state,
        domain_age,
        domain_end,
    ]], dtype=int)


In [ ]:
def RF_Model(X, y, X_test, y_test):
    """Tune Random Forest using 3-fold CV on training data, then evaluate once on holdout data."""
    print("________________ Random Forest __________________")

    param_grid = {
        "n_estimators": [200, 300],
        "max_depth": [10, None],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1],
    }

    # Label 1 is treated as the phishing class, so optimize its F1-score
    # rather than accuracy alone. The holdout set is not used during tuning.
    search = GridSearchCV(
        estimator=RandomForestClassifier(random_state=42, class_weight="balanced"),
        param_grid=param_grid,
        cv=3,
        scoring="f1",
        n_jobs=-1,
        verbose=1,
    )
    search.fit(X, y)
    model = search.best_estimator_

    print("Best parameters:", search.best_params_)
    print("Best cross-validation phishing F1:", round(search.best_score_, 4))

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(f"Holdout accuracy: {accuracy:.4f}")

    matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
    plt.figure(figsize=(6, 4))
    sns.heatmap(
        matrix,
        annot=True,
        cmap="YlGnBu",
        fmt="g",
        xticklabels=["Legitimate (0)", "Phishing (1)"],
        yticklabels=["Legitimate (0)", "Phishing (1)"],
    )
    plt.title("Confusion Matrix - Stratified Holdout")
    plt.ylabel("Actual label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.show()

    print("Classification Report:")
    print(classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["Legitimate", "Phishing"],
        zero_division=0,
    ))

    if len(np.unique(y_test)) == 2:
        positive_class_index = list(model.classes_).index(1)
        probabilities = model.predict_proba(X_test)[:, positive_class_index]
        auc = roc_auc_score(y_test, probabilities)
        fpr, tpr, _ = roc_curve(y_test, probabilities)
        print(f"Holdout ROC-AUC: {auc:.4f}")

        plt.figure(figsize=(6, 4))
        plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc:.2f})")
        plt.plot([0, 1], [0, 1], "--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve - Stratified Holdout")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.show()
    else:
        print("AUC/ROC skipped: evaluation data must contain both classes.")

    return model


def main():
    # Use one consistent source for both training and holdout evaluation.
    # This avoids comparing unrelated datasets with different feature distributions.
    training_data = importdata_train()
    X, y = splitdataset(training_data)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y,
    )

    print("Training rows:", len(X_train))
    print("Holdout rows:", len(X_test))
    print("Training class counts:", dict(zip(*np.unique(y_train, return_counts=True))))
    print("Holdout class counts:", dict(zip(*np.unique(y_test, return_counts=True))))

    # Tune and evaluate once; reuse this exact model in the GUI.
    model = RF_Model(X_train, y_train, X_test, y_test)

    class MainFrame(wx.Frame):
        def __init__(self):
            super().__init__(
                None,
                title="PhishTackle - Phishing URL Detector",
                size=(560, 360),
            )
            panel = wx.Panel(self)
            panel.SetBackgroundColour(wx.Colour("#F4F7FB"))

            layout = wx.BoxSizer(wx.VERTICAL)

            heading = wx.StaticText(panel, label="PhishTackle")
            heading.SetFont(wx.Font(
                22, wx.FONTFAMILY_DEFAULT, wx.FONTSTYLE_NORMAL, wx.FONTWEIGHT_BOLD
            ))
            heading.SetForegroundColour(wx.Colour("#173B67"))

            subtitle = wx.StaticText(
                panel,
                label="Check whether a website URL may be phishing.",
            )
            subtitle.SetForegroundColour(wx.Colour("#526477"))

            url_label = wx.StaticText(panel, label="Website URL")
            url_label.SetFont(wx.Font(
                10, wx.FONTFAMILY_DEFAULT, wx.FONTSTYLE_NORMAL, wx.FONTWEIGHT_BOLD
            ))

            self.url_input = wx.TextCtrl(
                panel,
                style=wx.TE_PROCESS_ENTER,
                size=(-1, 36),
            )
            self.url_input.SetHint("Example: https://www.example.com")

            button_row = wx.BoxSizer(wx.HORIZONTAL)
            scan_button = wx.Button(panel, label="Scan URL", size=(130, 38))
            clear_button = wx.Button(panel, label="Clear", size=(100, 38))
            button_row.Add(scan_button, 0, wx.RIGHT, 10)
            button_row.Add(clear_button, 0)

            self.result_label = wx.StaticText(
                panel,
                label="Result will appear here after scanning.",
                style=wx.ALIGN_CENTER,
            )
            self.result_label.SetFont(wx.Font(
                12, wx.FONTFAMILY_DEFAULT, wx.FONTSTYLE_NORMAL, wx.FONTWEIGHT_BOLD
            ))

            note = wx.StaticText(
                panel,
                label="Note: This is a machine-learning prediction, not a guarantee of safety.",
            )
            note.SetForegroundColour(wx.Colour("#667085"))

            layout.Add(heading, 0, wx.TOP | wx.ALIGN_CENTER, 24)
            layout.Add(subtitle, 0, wx.TOP | wx.BOTTOM | wx.ALIGN_CENTER, 8)
            layout.Add(url_label, 0, wx.LEFT | wx.RIGHT | wx.TOP, 28)
            layout.Add(self.url_input, 0, wx.LEFT | wx.RIGHT | wx.TOP | wx.EXPAND, 28)
            layout.Add(button_row, 0, wx.LEFT | wx.RIGHT | wx.TOP, 28)
            layout.Add(self.result_label, 0, wx.LEFT | wx.RIGHT | wx.TOP | wx.EXPAND, 28)
            layout.Add(note, 0, wx.ALL | wx.ALIGN_CENTER, 18)
            panel.SetSizer(layout)

            scan_button.Bind(wx.EVT_BUTTON, self.on_predict)
            clear_button.Bind(wx.EVT_BUTTON, self.on_clear)
            self.url_input.Bind(wx.EVT_TEXT_ENTER, self.on_predict)
            self.Centre()

        def on_predict(self, event):
            url = self.url_input.GetValue().strip()

            if not url:
                self.result_label.SetLabel("Please enter a website URL.")
                self.result_label.SetForegroundColour(wx.Colour("#B42318"))
                return

            try:
                features = extract_url_features(url)
                prediction = int(model.predict(features)[0])

                if prediction == 0:
                    self.result_label.SetLabel("LEGITIMATE - Model prediction")
                    self.result_label.SetForegroundColour(wx.Colour("#18794E"))
                else:
                    self.result_label.SetLabel("PHISHING - Model prediction")
                    self.result_label.SetForegroundColour(wx.Colour("#B42318"))

                self.Layout()
            except Exception as error:
                self.result_label.SetLabel(f"Unable to scan URL: {error}")
                self.result_label.SetForegroundColour(wx.Colour("#B42318"))
                self.Layout()

        def on_clear(self, event):
            self.url_input.Clear()
            self.result_label.SetLabel("Result will appear here after scanning.")
            self.result_label.SetForegroundColour(wx.Colour("#344054"))
            self.Layout()

    app = wx.App(False)
    frame = MainFrame()
    frame.Show()
    app.MainLoop()


if __name__ == "__main__":
    main()


## Note about `feature_test.csv`

`feature_test.csv` is intentionally not used for the headline accuracy or hyperparameter tuning because its feature distribution differs substantially from `training.xlsx` (for example, the `Domain_End` feature is constant in the training file but varies in the CSV). You can investigate that file separately as a domain-shift test, but its score should not be used as the main estimate of model performance. A reliable real-world estimate requires a representative, independently collected test set with the same feature definitions.